# Sistema de Reconocimiento Facial con Eigenfaces (PCA)
### Actividad 05 — Parte b

**Objetivo:** construir un sistema de **reconocimiento facial por identificación** usando el dataset
*Olivetti Faces* de scikit-learn. Dado el rostro de una persona (que el modelo no vio en
entrenamiento), el sistema debe decir a cuál de las 40 personas conocidas pertenece.

**Restricción de la actividad:** no se permite usar redes neuronales. Se usa el método clásico
**Eigenfaces (Turk & Pentland, 1991)**:

1. Cada imagen de rostro se aplana en un vector de píxeles (4096 dimensiones).
2. **Preprocesamiento:** se calcula el **rostro promedio** y se centra cada imagen restándolo,
   dejando los datos centrados en el origen antes de aplicar PCA.
3. **PCA** aprende las direcciones (eigenvectores) de mayor varianza entre todos los rostros de
   entrenamiento. Esas direcciones, reconstruidas como imágenes, son las *eigenfaces* — son
   literalmente el atributo `components_` de PCA.
4. Cada rostro se re-expresa con pocos coeficientes (sus "pesos" sobre las eigenfaces) en vez de
   miles de píxeles — esto es la reducción de dimensión / "espacio de caras".
5. Un clasificador **no neuronal** decide la identidad comparando esos coeficientes en el
   espacio reducido. Se implementan y comparan dos opciones clásicas:
   - **SVM (Support Vector Machine)** — clasificador principal de este notebook, recomendado por
     su robustez ante variabilidad de iluminación, pose y expresión.
   - **KNN (K-Nearest Neighbors)** — clasificador de referencia, más simple e interpretable.

**Reconocimiento vs. detección vs. verificación:**

| Término | Pregunta que responde |
|---|---|
| Detección facial | ¿Hay una cara en la imagen y dónde está? |
| **Reconocimiento (identificación)** — *esto hacemos aquí* | Dado un rostro ya recortado, ¿de cuál de las N personas conocidas es? |
| Verificación | ¿Este rostro es la misma persona que dice ser? (1 contra 1) |

## 1. Importar librerías

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_olivetti_faces
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, classification_report,
                              confusion_matrix, ConfusionMatrixDisplay)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 2. Cargar el dataset Olivetti Faces

- 400 imágenes en total: **40 personas × 10 fotos cada una**.
- Cada imagen es de **64×64 píxeles** en escala de grises (igual concepto que en tu documento
  de NMF: "Imágenes a escala de grises" se representan como matrices con valores entre 0 y 1).
- `faces.data` ya viene **aplanado** (flattened): forma `(400, 4096)`, tal como se explica en la
  sección "Imágenes en escala de grises como matrices planas" de tu apunte de NMF.
- `faces.target` trae la etiqueta de la persona (0 a 39).


In [ ]:
faces = fetch_olivetti_faces(shuffle=True, random_state=RANDOM_STATE)
X, y = faces.data, faces.target
h, w = faces.images.shape[1], faces.images.shape[2]  # 64, 64

n_samples, n_features = X.shape
n_classes = len(np.unique(y))

print(f"Muestras: {n_samples}")
print(f"Características (píxeles) por imagen: {n_features}")
print(f"Número de personas (clases): {n_classes}")
print(f"Dimensión de cada imagen: {h}x{w}")


### Vista previa de algunos rostros del dataset

In [ ]:
fig, axes = plt.subplots(2, 8, figsize=(14, 4),
                          subplot_kw={'xticks': [], 'yticks': []})
for ax, i in zip(axes.ravel(), np.random.choice(n_samples, 16, replace=False)):
    ax.imshow(faces.images[i], cmap='gray')
    ax.set_title(f"ID {y[i]}", fontsize=9)
plt.suptitle("Muestra de rostros del dataset Olivetti Faces")
plt.tight_layout()
plt.show()


## 3. Separar entrenamiento y prueba

Se separa **por muestra**, no por persona: cada una de las 40 personas debe tener fotos tanto en
entrenamiento como en prueba, para poder evaluar si el sistema reconoce correctamente rostros de
esa persona que no vio antes. Se usa `stratify=y` para mantener la proporción de cada clase.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE
)

print(f"Entrenamiento: {X_train.shape[0]} imágenes")
print(f"Prueba:        {X_test.shape[0]} imágenes")


## 4. Preprocesamiento y PCA — Aprender las "Eigenfaces"

### 4.0 Centrado de datos: el "rostro promedio"

Antes de aplicar PCA, se calcula el **rostro promedio** (la media de todos los vectores faciales
de entrenamiento) y se resta de cada imagen individual. Esto centra los datos en el origen, que
es un requisito matemático de PCA (los componentes principales se calculan sobre la matriz de
covarianza de datos centrados). En scikit-learn, `PCA.fit()` hace este centrado automáticamente
de forma interna, pero aquí lo mostramos explícitamente para visualizar el rostro promedio y
entender qué hace el algoritmo por dentro.

Los valores de píxeles del dataset Olivetti ya vienen escalados en el rango [0, 1] (float32), por
lo que no se requiere un escalado adicional tipo `StandardScaler`; con el centrado y `whiten=True`
en PCA es suficiente para que ningún componente domine por tener valores más grandes.

Igual que en tu documento de PCA: `fit()` **aprende** la rotación/desplazamiento sin modificar
los datos; `transform()` **aplica** esa transformación.

**Importante:** el rostro promedio y `pca.fit()` se calculan **solo con datos de entrenamiento**
para no filtrar información del conjunto de prueba (data leakage).

In [ ]:
# Rostro promedio: media de todos los vectores de entrenamiento (centrado de datos)
mean_face = X_train.mean(axis=0)

plt.figure(figsize=(3, 3))
plt.imshow(mean_face.reshape(h, w), cmap='gray')
plt.title("Rostro promedio (entrenamiento)")
plt.xticks([]); plt.yticks([])
plt.show()

# PCA internamente resta esta misma media antes de calcular los componentes;
# aquí solo la visualizamos para fines didácticos.
print("Forma del rostro promedio:", mean_face.shape)

### 4.1 Varianza explicada — ¿cuántos componentes conservar?

Antes de fijar el número final de componentes, se ajusta un PCA completo (todas las
componentes posibles) sobre el conjunto de entrenamiento y se observa la varianza acumulada
explicada. Esto permite decidir con evidencia cuántas eigenfaces son necesarias para retener
un porcentaje objetivo de la información original, en vez de fijar un número arbitrario.

Para el dataset Olivetti se reporta en la literatura que entre 30 y 50 componentes ya explican
cerca del 95% de la varianza; a continuación se verifica ese rango con los datos reales de
este notebook.

In [ ]:
pca_full = PCA(random_state=RANDOM_STATE).fit(X_train)
cum_var = np.cumsum(pca_full.explained_variance_ratio_)

plt.figure(figsize=(7, 4))
plt.plot(cum_var)
plt.axhline(0.95, color='red', linestyle='--', label='95% de varianza')
plt.xlabel('Número de componentes PCA')
plt.ylabel('Varianza acumulada explicada')
plt.title('Varianza explicada acumulada')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

n_95 = np.argmax(cum_var >= 0.95) + 1
print(f"Componentes necesarios para explicar el 95% de la varianza: {n_95}")
print(f"Varianza explicada con 30 componentes: {cum_var[29]:.4f}")
print(f"Varianza explicada con 50 componentes: {cum_var[49]:.4f}")
print(f"Varianza explicada con 150 componentes: {cum_var[149]:.4f}")

### 4.2 Elegir el número de componentes y entrenar el PCA final

Con la curva de varianza explicada como evidencia, se fija `N_COMPONENTS`. Se documentan dos
criterios posibles:

- **Criterio compacto (30–50 componentes):** suficiente para retener ~95% de la varianza en
  Olivetti Faces, con máxima compresión y menor costo computacional.
- **Criterio conservador (150 componentes):** margen adicional de información, útil cuando se
  quiere maximizar la precisión del clasificador aun a costa de menor compresión.

Este notebook usa el valor calculado automáticamente a partir del 95% de varianza explicada
(dentro del rango 30–50 esperado para este dataset), lo que cumple con el criterio de
compresión inteligente sin perder información esencial para el reconocimiento.

**Importante:** `pca.fit()` se ajusta **solo con datos de entrenamiento** para no filtrar
información del conjunto de prueba (data leakage).

In [ ]:
# Se fija el número de componentes según el 95% de varianza explicada (n_95),
# acotado al rango esperado para Olivetti Faces documentado en la literatura (30-50).
N_COMPONENTS = int(np.clip(n_95, 30, 50))

pca = PCA(n_components=N_COMPONENTS, whiten=True, random_state=RANDOM_STATE)
pca.fit(X_train)

print(f"N_COMPONENTS elegido: {N_COMPONENTS}")
print("Forma de components_:", pca.components_.shape)
print(f"Varianza explicada con este N_COMPONENTS: {cum_var[N_COMPONENTS-1]:.4f}")

### 4.3 Visualizar las Eigenfaces

Cada fila de `pca.components_` es un vector de 4096 números — se puede volver a dar forma
(`reshape`) a 64x64 para visualizarla como imagen. Estas imágenes "fantasmales" son las
direcciones principales de variación entre rostros: las primeras suelen capturar iluminación
general y contraste global, mientras que componentes posteriores capturan rasgos más finos
(forma de ojos, nariz, cejas, anteojos, vello facial).

In [ ]:
eigenfaces = pca.components_.reshape((N_COMPONENTS, h, w))

n_show = min(16, N_COMPONENTS)
fig, axes = plt.subplots(2, 8, figsize=(14, 4),
                          subplot_kw={'xticks': [], 'yticks': []})
for i, ax in enumerate(axes.ravel()):
    if i < n_show:
        ax.imshow(eigenfaces[i], cmap='gray')
        ax.set_title(f"Eigenface {i+1}", fontsize=8)
    else:
        ax.axis('off')
plt.suptitle("Primeras eigenfaces (componentes principales)")
plt.tight_layout()
plt.show()

## 5. Proyectar los rostros al espacio de Eigenfaces

Se transforman entrenamiento y prueba: cada rostro pasa de 4096 píxeles a solo `N_COMPONENTS`
coordenadas (sus "pesos" sobre cada eigenface).


In [ ]:
X_train_pca = pca.transform(X_train)
X_test_pca = pca.transform(X_test)

print("Forma original:", X_train.shape)
print("Forma reducida (proyección en eigenfaces):", X_train_pca.shape)


### 5.1 Reconstrucción — verificar que no se pierde información importante

Al igual que en la sección de "Reconstrucción de una muestra" de tu documento de NMF (multiplicar
componentes por pesos y sumar), aquí se puede reconstruir aproximadamente el rostro original a
partir de sus coordenadas PCA, para comprobar visualmente que las eigenfaces capturan lo esencial.


In [ ]:
def reconstruir(pca_model, X_proj, idx):
    # PCA con whiten deshace el escalado internamente en inverse_transform
    return pca_model.inverse_transform(X_proj[idx].reshape(1, -1)).reshape(h, w)

fig, axes = plt.subplots(2, 6, figsize=(12, 4.5),
                          subplot_kw={'xticks': [], 'yticks': []})
idxs = np.random.choice(X_test.shape[0], 6, replace=False)
for col, i in enumerate(idxs):
    axes[0, col].imshow(X_test[i].reshape(h, w), cmap='gray')
    axes[0, col].set_title("Original", fontsize=8)
    axes[1, col].imshow(reconstruir(pca, X_test_pca, i), cmap='gray')
    axes[1, col].set_title("Reconstruida", fontsize=8)
plt.suptitle(f"Original vs. Reconstrucción usando {N_COMPONENTS} eigenfaces")
plt.tight_layout()
plt.show()


## 6. Clasificadores — SVM y KNN (sin redes neuronales)

Con los rostros ya representados por sus coeficientes en el espacio de eigenfaces, se entrenan
y comparan **dos clasificadores supervisados clásicos**, ninguno de los cuales es una red
neuronal:

- **SVM (Support Vector Machine)** — clasificador **principal** de este notebook. Busca
  hiperplanos óptimos que maximizan el margen entre identidades; se usa kernel RBF, que suele
  ser más robusto que un kernel lineal ante la variabilidad de iluminación, pose y expresión
  entre fotos de una misma persona.
- **KNN (K-Nearest Neighbors)** — clasificador de **referencia/comparación**. Clasifica un
  rostro nuevo buscando la identidad más cercana en el espacio reducido, usando distancia
  euclidiana (u otras métricas).

Ambos son más interpretables y requieren muchísima menos potencia computacional que una red
neuronal, cumpliendo la restricción de la actividad.

Se prueban varios hiperparámetros para cada uno con `GridSearchCV` y validación cruzada (cv=5)
para elegir la mejor configuración de forma objetiva.

In [ ]:
# --- SVM (clasificador principal) ---
param_grid_svm = {
    'C': [1, 10, 100, 1000],
    'gamma': [0.0001, 0.001, 0.01, 0.1],
    'kernel': ['rbf'],
}

grid_svm = GridSearchCV(
    SVC(class_weight='balanced', random_state=RANDOM_STATE),
    param_grid_svm, cv=5, scoring='accuracy'
)
grid_svm.fit(X_train_pca, y_train)

print("SVM — Mejores parámetros:", grid_svm.best_params_)
print(f"SVM — Accuracy en validación cruzada: {grid_svm.best_score_:.4f}")

clf_svm = grid_svm.best_estimator_

### 6.1 KNN — clasificador de comparación

Se entrena también un KNN con búsqueda de hiperparámetros, para poder comparar su desempeño
contra el SVM en la sección de evaluación.

In [ ]:
# --- KNN (clasificador de comparación) ---
param_grid_knn = {'n_neighbors': [1, 3, 5, 7, 9], 'metric': ['euclidean', 'manhattan', 'cosine']}

grid_knn = GridSearchCV(KNeighborsClassifier(), param_grid_knn, cv=5, scoring='accuracy')
grid_knn.fit(X_train_pca, y_train)

print("KNN — Mejores parámetros:", grid_knn.best_params_)
print(f"KNN — Accuracy en validación cruzada: {grid_knn.best_score_:.4f}")

clf_knn = grid_knn.best_estimator_

# Clasificador principal usado en el resto del notebook
clf = clf_svm

## 7. Evaluación en el conjunto de prueba (rostros nunca vistos)

Se evalúan **ambos clasificadores** (SVM y KNN) sobre el conjunto de prueba, que contiene
rostros que el modelo no vio durante el entrenamiento. Esto permite comparar de forma empírica
cuál generaliza mejor para este dataset.

In [ ]:
y_pred_svm = clf_svm.predict(X_test_pca)
y_pred_knn = clf_knn.predict(X_test_pca)

acc_svm = accuracy_score(y_test, y_pred_svm)
acc_knn = accuracy_score(y_test, y_pred_knn)

print(f"SVM — Accuracy en test: {acc_svm:.4f}")
print(f"KNN — Accuracy en test: {acc_knn:.4f}")
print()
print("=== Reporte de clasificación — SVM ===")
print(classification_report(y_test, y_pred_svm, zero_division=0))
print("=== Reporte de clasificación — KNN ===")
print(classification_report(y_test, y_pred_knn, zero_division=0))

# Variables usadas en el resto del notebook (predicciones del clasificador principal: SVM)
y_pred = y_pred_svm

### 7.1 Matrices de confusión — SVM vs. KNN

La matriz de confusión muestra qué personas son confundidas entre sí por cada clasificador.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 9))

cm_svm = confusion_matrix(y_test, y_pred_svm)
ConfusionMatrixDisplay(cm_svm).plot(ax=axes[0], cmap='Blues', colorbar=False)
axes[0].set_title(f"SVM (accuracy = {acc_svm:.4f})")

cm_knn = confusion_matrix(y_test, y_pred_knn)
ConfusionMatrixDisplay(cm_knn).plot(ax=axes[1], cmap='Oranges', colorbar=False)
axes[1].set_title(f"KNN (accuracy = {acc_knn:.4f})")

plt.suptitle("Matrices de confusión (40 personas): SVM vs. KNN")
plt.tight_layout()
plt.show()

### 7.2 Comparación resumida SVM vs. KNN

Tabla resumen para incluir en el informe, comparando ambos clasificadores en accuracy de
validación cruzada y de test.

In [ ]:
import pandas as pd

resumen = pd.DataFrame({
    'Clasificador': ['SVM (RBF)', 'KNN'],
    'Mejor configuración': [str(grid_svm.best_params_), str(grid_knn.best_params_)],
    'Accuracy CV (train)': [grid_svm.best_score_, grid_knn.best_score_],
    'Accuracy Test': [acc_svm, acc_knn],
})
resumen

### 7.3 Ejemplos de predicciones — aciertos y errores (SVM)

Inspeccionar visualmente algunos casos, sobre todo los errores, ayuda a explicar los hallazgos
(por ejemplo, confusiones entre personas con anteojos, poses similares o iluminación parecida).
Se muestran los resultados del clasificador principal (SVM).

In [ ]:
def mostrar_predicciones(idxs, titulo):
    fig, axes = plt.subplots(1, len(idxs), figsize=(2.2*len(idxs), 3),
                              subplot_kw={'xticks': [], 'yticks': []})
    if len(idxs) == 1:
        axes = [axes]
    for ax, i in zip(axes, idxs):
        ax.imshow(X_test[i].reshape(h, w), cmap='gray')
        color = 'green' if y_pred[i] == y_test[i] else 'red'
        ax.set_title(f"Real:{y_test[i]}\nPred:{y_pred[i]}", color=color, fontsize=9)
    plt.suptitle(titulo)
    plt.tight_layout()
    plt.show()

aciertos = np.where(y_pred == y_test)[0]
errores = np.where(y_pred != y_test)[0]

mostrar_predicciones(np.random.choice(aciertos, min(6, len(aciertos)), replace=False),
                      "Ejemplos correctamente reconocidos")

if len(errores) > 0:
    mostrar_predicciones(errores[:min(6, len(errores))], "Ejemplos mal reconocidos")
else:
    print("No hubo errores de clasificación en el conjunto de prueba.")


## 8. Conclusiones (para incluir en el informe)

Completa esta celda con tus resultados reales una vez ejecutes el notebook con internet
disponible. Puntos a comentar:

- **Accuracy final** de SVM y de KNN en el conjunto de prueba, comparados entre sí y contra el
  accuracy de validación cruzada de cada uno (¿ambos modelos generalizan bien a rostros nuevos?
  ¿cuál generaliza mejor?).
- **Cuál clasificador conviene usar** para este sistema y por qué (robustez ante variabilidad
  de iluminación/pose para SVM vs. simplicidad e interpretabilidad geométrica de KNN).
- **Cuántos componentes PCA (eigenfaces)** fueron necesarios para el 95% de varianza, y cómo se
  compara ese número con el rango típico documentado (30–50 componentes) para Olivetti Faces.
- **Qué muestran las primeras eigenfaces** (suelen capturar iluminación general y contorno de
  cara en los primeros componentes, y detalles más finos como anteojos o vello facial en
  componentes posteriores).
- **Qué tipo de errores comete cada modelo** (personas con gestos muy distintos entre fotos,
  anteojos que aparecen solo en algunas tomas, iluminación extrema, etc.) y si ambos
  clasificadores se equivocan en los mismos casos o en casos distintos.
- Recuerda explicar que este es un sistema de **reconocimiento por identificación** (no
  detección ni verificación) y que no se usó ninguna red neuronal — se usó PCA (Eigenfaces)
  combinado con dos clasificadores clásicos: SVM (principal) y KNN (comparación).